In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Importar módulos propios
from src.preprocessing import preprocess_for_tfidf
from src.augmentation import augment_text
from src.evaluation import (
    get_metrics,
    compare_train_test,
    display_comparison,
    print_overfitting_analysis,
    print_report
)

# Fijar semillas para reproducibilidad
random.seed(42)
np.random.seed(42)

In [2]:
# Descargar recursos de NLTK
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("✅ Recursos NLTK descargados")

✅ Recursos NLTK descargados


## 1. Cargar y preparar datos

In [3]:
# Cargar dataset limpio
df = pd.read_csv('../data/processed/youtoxic_clean.csv')
print(f'Total de registros: {len(df)}')
print(f'Distribución de clases:')
print(df['IsToxic'].value_counts())

Total de registros: 1000
Distribución de clases:
IsToxic
False    538
True     462
Name: count, dtype: int64


In [4]:
# Separar variables y dividir train/test
X = df['Text']
y = df['IsToxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

Train: 800, Test: 200


## 2. Preprocesamiento

In [5]:
# Aplicar preprocesamiento
print('Preprocesando textos...')
X_train_processed = X_train.apply(preprocess_for_tfidf)
X_test_processed = X_test.apply(preprocess_for_tfidf)
print(f'✅ Preprocesamiento completado')

# Ejemplo
print(f'\nEjemplo:')
print(f'Original: {X_train.iloc[0][:100]}...')
print(f'Procesado: {X_train_processed.iloc[0][:100]}...')

Preprocesando textos...
✅ Preprocesamiento completado

Ejemplo:
Original: i wonder what the police expect will happen when they continually abuse and kill people do they hone...
Procesado: wonder police expect happen continually abuse kill people honestly expect public nothing lay back ta...
✅ Preprocesamiento completado

Ejemplo:
Original: i wonder what the police expect will happen when they continually abuse and kill people do they hone...
Procesado: wonder police expect happen continually abuse kill people honestly expect public nothing lay back ta...


## 3. Data Augmentation (solo en Train)

In [6]:
# Aplicar augmentation 2X al train
train_df = pd.DataFrame({'Text': X_train_processed, 'IsToxic': y_train})

print(f'Train original: {len(train_df)}')

augmented_texts = []
augmented_labels = []

print('Aplicando augmentation 2X...')
for i, (text, label) in enumerate(zip(train_df['Text'], train_df['IsToxic'])):
    augmented_texts.append(augment_text(text))
    augmented_labels.append(label)
    augmented_texts.append(augment_text(text))
    augmented_labels.append(label)
    if (i + 1) % 200 == 0:
        print(f'  Procesados {i + 1}/{len(train_df)}...')

# Combinar original + augmentados
df_augmented = pd.DataFrame({'Text': augmented_texts, 'IsToxic': augmented_labels})
train_final = pd.concat([train_df, df_augmented], ignore_index=True)
train_final = train_final.sample(frac=1, random_state=42).reset_index(drop=True)

X_train_aug = train_final['Text']
y_train_aug = train_final['IsToxic']

print(f'\nTrain después de augmentation: {len(train_final)} (3x original)')

Train original: 800
Aplicando augmentation 2X...
  Procesados 200/800...
  Procesados 200/800...
  Procesados 400/800...
  Procesados 400/800...
  Procesados 600/800...
  Procesados 600/800...
  Procesados 800/800...

Train después de augmentation: 2400 (3x original)
  Procesados 800/800...

Train después de augmentation: 2400 (3x original)


## 4. Vectorización TF-IDF

In [7]:
# Vectorizar con TF-IDF
vectorizer = TfidfVectorizer(
    max_features=500,
    min_df=3,
    max_df=0.90,
    ngram_range=(1, 1)
)

X_train_tfidf = vectorizer.fit_transform(X_train_aug)
X_test_tfidf = vectorizer.transform(X_test_processed)

print(f'Shape train: {X_train_tfidf.shape}')
print(f'Shape test: {X_test_tfidf.shape}')
print(f'Vocabulario: {len(vectorizer.vocabulary_)} palabras')

Shape train: (2400, 500)
Shape test: (200, 500)
Vocabulario: 500 palabras


## 5. Entrenar Naive Bayes

Usamos `MultinomialNB` que es ideal para features de conteo/frecuencia como TF-IDF.

**Parámetro `alpha`:** Suavizado de Laplace para evitar probabilidades cero.

In [8]:
# Entrenar Naive Bayes
clf_nb = MultinomialNB(alpha=1.0)  # alpha = suavizado Laplace

clf_nb.fit(X_train_tfidf, y_train_aug)
print(f'✅ Modelo Naive Bayes entrenado con {len(y_train_aug)} muestras')
print(f'   Suavizado alpha = {clf_nb.alpha}')

✅ Modelo Naive Bayes entrenado con 2400 muestras
   Suavizado alpha = 1.0


## 6. Evaluación en Train

In [9]:
# Predicciones en train
y_train_pred = clf_nb.predict(X_train_tfidf)

# Métricas
metrics_train = get_metrics(y_train_aug, y_train_pred)
print('--- Métricas en Train ---')
for metric, value in metrics_train.items():
    print(f'{metric.capitalize()}: {value:.4f}')

print_report(y_train_aug, y_train_pred, "Naive Bayes - Train")

--- Métricas en Train ---
Accuracy: 0.8321
Precision: 0.8462
Recall: 0.7784
F1: 0.8109

--- Naive Bayes - Train ---
              precision    recall  f1-score   support

       False       0.82      0.88      0.85      1290
        True       0.85      0.78      0.81      1110

    accuracy                           0.83      2400
   macro avg       0.83      0.83      0.83      2400
weighted avg       0.83      0.83      0.83      2400



## 7. Evaluación en Test

In [10]:
# Predicciones en test
y_pred = clf_nb.predict(X_test_tfidf)

# Métricas
metrics_test = get_metrics(y_test, y_pred)
print('--- Métricas en Test ---')
for metric, value in metrics_test.items():
    print(f'{metric.capitalize()}: {value:.4f}')

print_report(y_test, y_pred, "Naive Bayes - Test")

--- Métricas en Test ---
Accuracy: 0.6950
Precision: 0.7067
Recall: 0.5761
F1: 0.6347

--- Naive Bayes - Test ---
              precision    recall  f1-score   support

       False       0.69      0.80      0.74       108
        True       0.71      0.58      0.63        92

    accuracy                           0.69       200
   macro avg       0.70      0.69      0.69       200
weighted avg       0.70      0.69      0.69       200



## 8. Análisis de Overfitting

In [11]:
# Comparar train vs test
comparison = compare_train_test(y_train_aug, y_train_pred, y_test, y_pred)

print('📊 Comparativa Train vs Test:')
display(display_comparison(comparison))

print_overfitting_analysis(comparison)

📊 Comparativa Train vs Test:


,accuracy,precision,recall,f1
Train,0.832083,0.846229,0.778378,0.810887
Test,0.695000,0.706667,0.576087,0.634731
Gap (Train-Test),0.137083,0.139563,0.202291,0.176156



🎯 Análisis de Overfitting:
⚠️ Gap F1 = 0.176 (>5%) - Posible overfitting


## 9. Comparativa con Baseline Trivial

In [12]:
# Cargar baseline trivial
df_baseline = pd.read_csv('../data/processed/baseline_trivial_pred.csv')
y_true_baseline = df_baseline['IsToxic']
y_pred_baseline = df_baseline['baseline_pred']

metrics_baseline = get_metrics(y_true_baseline, y_pred_baseline)
metrics_nb = get_metrics(y_test, y_pred)

# Tabla comparativa
report = pd.DataFrame(
    [metrics_baseline, metrics_nb],
    index=['Baseline Trivial', 'TF-IDF + Naive Bayes']
)
print('Comparativa de métricas:')
display(report)

Comparativa de métricas:


,accuracy,precision,recall,f1
Baseline Trivial,0.538,0.000000,0.000000,0.000000
TF-IDF + Naive Bayes,0.695,0.706667,0.576087,0.634731


## 10. Comparativa con Logistic Regression

Resultados de referencia del notebook anterior:
- **Logistic Regression:** F1 Test ≈ 0.686, Gap ≈ 9.2%

In [13]:
# Comparativa final
print('=' * 50)
print('RESUMEN COMPARATIVO')
print('=' * 50)
print(f'\nNaive Bayes:')
print(f'  - F1 Test: {metrics_test["f1"]:.4f}')
print(f'  - Gap F1: {comparison["gap_f1"]:.4f} ({comparison["gap_f1"]*100:.1f}%)')
print(f'\nLogistic Regression (referencia):')
print(f'  - F1 Test: ~0.686')
print(f'  - Gap F1: ~0.092 (9.2%)')
print('\n' + '=' * 50)

RESUMEN COMPARATIVO

Naive Bayes:
  - F1 Test: 0.6347
  - Gap F1: 0.1762 (17.6%)

Logistic Regression (referencia):
  - F1 Test: ~0.686
  - Gap F1: ~0.092 (9.2%)



## 11. Optimización de Naive Bayes

El modelo base tiene mucho overfitting (17.6%). Vamos a probar:
1. **Diferentes valores de alpha** (suavizado Laplace)
2. **ComplementNB** (mejor para clases desbalanceadas)
3. **Reducir features** del TF-IDF

In [14]:
# Grid Search manual para encontrar mejor configuración
from sklearn.naive_bayes import ComplementNB

# Probar diferentes valores de alpha
alphas = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
results = []

print("🔍 Probando diferentes valores de alpha (MultinomialNB):")
print("-" * 60)

for alpha in alphas:
    clf = MultinomialNB(alpha=alpha)
    clf.fit(X_train_tfidf, y_train_aug)
    
    y_train_p = clf.predict(X_train_tfidf)
    y_test_p = clf.predict(X_test_tfidf)
    
    f1_train = get_metrics(y_train_aug, y_train_p)['f1']
    f1_test = get_metrics(y_test, y_test_p)['f1']
    gap = f1_train - f1_test
    
    results.append({
        'model': 'MultinomialNB',
        'alpha': alpha,
        'f1_train': f1_train,
        'f1_test': f1_test,
        'gap': gap
    })
    print(f"  alpha={alpha:>5}: F1 Train={f1_train:.4f}, F1 Test={f1_test:.4f}, Gap={gap:.4f}")

# Probar ComplementNB
print("\n🔍 Probando ComplementNB (mejor para clases desbalanceadas):")
print("-" * 60)

for alpha in alphas:
    clf = ComplementNB(alpha=alpha)
    clf.fit(X_train_tfidf, y_train_aug)
    
    y_train_p = clf.predict(X_train_tfidf)
    y_test_p = clf.predict(X_test_tfidf)
    
    f1_train = get_metrics(y_train_aug, y_train_p)['f1']
    f1_test = get_metrics(y_test, y_test_p)['f1']
    gap = f1_train - f1_test
    
    results.append({
        'model': 'ComplementNB',
        'alpha': alpha,
        'f1_train': f1_train,
        'f1_test': f1_test,
        'gap': gap
    })
    print(f"  alpha={alpha:>5}: F1 Train={f1_train:.4f}, F1 Test={f1_test:.4f}, Gap={gap:.4f}")

🔍 Probando diferentes valores de alpha (MultinomialNB):
------------------------------------------------------------
  alpha= 0.01: F1 Train=0.8239, F1 Test=0.6303, Gap=0.1936
  alpha=  0.1: F1 Train=0.8239, F1 Test=0.6386, Gap=0.1854
  alpha=  0.5: F1 Train=0.8185, F1 Test=0.6347, Gap=0.1838
  alpha=  1.0: F1 Train=0.8109, F1 Test=0.6347, Gap=0.1762
  alpha=  2.0: F1 Train=0.8104, F1 Test=0.6341, Gap=0.1763
  alpha=  5.0: F1 Train=0.7889, F1 Test=0.5949, Gap=0.1940
  alpha= 10.0: F1 Train=0.7675, F1 Test=0.6013, Gap=0.1662

🔍 Probando ComplementNB (mejor para clases desbalanceadas):
------------------------------------------------------------
  alpha= 0.01: F1 Train=0.8174, F1 Test=0.6556, Gap=0.1619
  alpha=  0.1: F1 Train=0.8185, F1 Test=0.6556, Gap=0.1629
  alpha=  0.5: F1 Train=0.8185, F1 Test=0.6592, Gap=0.1592
  alpha=  1.0: F1 Train=0.8209, F1 Test=0.6629, Gap=0.1580
  alpha=  2.0: F1 Train=0.8217, F1 Test=0.6816, Gap=0.1401
  alpha=  5.0: F1 Train=0.8155, F1 Test=0.6667, Gap=0

In [15]:
# Encontrar mejor configuración
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('f1_test', ascending=False)

print("\n📊 Top 5 configuraciones (ordenadas por F1 Test):")
display(results_df.head(10))

# Mejor modelo
best = results_df.iloc[0]
print(f"\n🏆 Mejor configuración:")
print(f"   Modelo: {best['model']}")
print(f"   Alpha: {best['alpha']}")
print(f"   F1 Test: {best['f1_test']:.4f}")
print(f"   Gap: {best['gap']:.4f} ({best['gap']*100:.1f}%)")


📊 Top 5 configuraciones (ordenadas por F1 Test):


,model,alpha,f1_train,f1_test,gap
11,ComplementNB,2.00,0.821650,0.681564,0.140086
13,ComplementNB,10.00,0.805160,0.681319,0.123841
12,ComplementNB,5.00,0.815534,0.666667,0.148867
10,ComplementNB,1.00,0.820922,0.662921,0.158001
9,ComplementNB,0.50,0.818464,0.659218,0.159246
7,ComplementNB,0.01,0.817414,0.655556,0.161859
8,ComplementNB,0.10,0.818505,0.655556,0.162950
1,MultinomialNB,0.10,0.823914,0.638554,0.185360
2,MultinomialNB,0.50,0.818522,0.634731,0.183791
3,MultinomialNB,1.00,0.810887,0.634731,0.176156



🏆 Mejor configuración:
   Modelo: ComplementNB
   Alpha: 2.0
   F1 Test: 0.6816
   Gap: 0.1401 (14.0%)


In [ ]:
# Entrenar modelo final con mejor configuración
best_model_type = best['model']
best_alpha = best['alpha']

if best_model_type == 'MultinomialNB':
    clf_best = MultinomialNB(alpha=best_alpha)
else:
    clf_best = ComplementNB(alpha=best_alpha)

clf_best.fit(X_train_tfidf, y_train_aug)

# Predicciones finales
y_train_pred_best = clf_best.predict(X_train_tfidf)
y_pred_best = clf_best.predict(X_test_tfidf)

# Métricas finales
print(f"Modelo optimizado: {best_model_type} (alpha={best_alpha})")
print("\n--- Métricas en Test (Optimizado) ---")
metrics_best = get_metrics(y_test, y_pred_best)
for metric, value in metrics_best.items():
    print(f'{metric.capitalize()}: {value:.4f}')

print_report(y_test, y_pred_best, f"{best_model_type} Optimizado - Test")

✅ Modelo optimizado: ComplementNB (alpha=2.0)

--- Métricas en Test (Optimizado) ---
Accuracy: 0.7150
Precision: 0.7011
Recall: 0.6630
F1: 0.6816

--- ComplementNB Optimizado - Test ---
              precision    recall  f1-score   support

       False       0.73      0.76      0.74       108
        True       0.70      0.66      0.68        92

    accuracy                           0.71       200
   macro avg       0.71      0.71      0.71       200
weighted avg       0.71      0.71      0.71       200



In [ ]:
# Comparativa final: Baseline vs NB Original vs NB Optimizado vs LogReg
comparison_best = compare_train_test(y_train_aug, y_train_pred_best, y_test, y_pred_best)

print("=" * 60)
print("COMPARATIVA FINAL")
print("=" * 60)

final_comparison = pd.DataFrame([
    {'Modelo': 'Baseline Trivial', 'F1 Test': 0.0, 'Gap F1': 'N/A'},
    {'Modelo': 'Naive Bayes (alpha=1.0)', 'F1 Test': 0.635, 'Gap F1': '17.6%'},
    {'Modelo': f'{best_model_type} (alpha={best_alpha})', 'F1 Test': metrics_best['f1'], 'Gap F1': f"{comparison_best['gap_f1']*100:.1f}%"},
    {'Modelo': 'Logistic Regression (C=0.005)', 'F1 Test': 0.686, 'Gap F1': '9.2%'}
])

display(final_comparison)

print("\nAnálisis:")
if metrics_best['f1'] > 0.686:
    print(f"Naive Bayes optimizado SUPERA a Logistic Regression!")
else:
    print(f"Logistic Regression sigue siendo mejor para este dataset")
print(f"   Gap reducido de 17.6% a {comparison_best['gap_f1']*100:.1f}%")

COMPARATIVA FINAL


,Modelo,F1 Test,Gap F1
0,Baseline Trivial,0.000000,N/A
1,Naive Bayes (alpha=1.0),0.635000,17.6%
2,ComplementNB (alpha=2.0),0.681564,14.0%
3,Logistic Regression (C=0.005),0.686000,9.2%



🎯 Análisis:
📊 Logistic Regression sigue siendo mejor para este dataset
   Gap reducido de 17.6% a 14.0%
